# Chapter 08 · Notebook 01: Meet the Adam optimizer

[Chapter guide](README.md) · [Previous chapter](../07_freezing_parameters/README.md) · [Course index](../README.md)

We know SGD and momentum. Adam is another optimizer that uses stored information about gradients and their size to choose parameter updates. We will first observe what it does in a small experiment, then explain its behavior in everyday words.

The training sequence stays familiar: clear gradients, make predictions, measure loss, calculate gradients, and update parameters. The optimizer changes how the update is chosen; our model and data stay familiar.

Start in a fresh kernel. Work through one lesson at a time with the mentoring conversation. This notebook supplies its own data and model instances.

## Contents

1. [Compare the first SGD and Adam updates](#first-update)
2. [Explain the first Adam step with small numbers](#first-step-size)
3. [Inspect Adam's stored history](#adam-memory)
4. [Build the remembered gradient average](#gradient-average)
5. [Build the remembered squared-gradient average](#squared-average)
6. [Perform the actual second Adam update](#second-update)
7. [Connect the first-step memories to the parameter change](#memory-to-step)
8. [Compare longer training and validation runs](#longer-comparison)

## Setup: our familiar one-input prediction problem

Use inputs `[0, 1, 2, 3]` and targets `[2, 5, 8, 11]`. Each input has one feature. Arrange them as `[4, 1]` tensors for a `Linear(1, 1)` layer. We will initialize both models to weight `1` and bias `0` in the lesson cell.


In [ ]:
import torch

# Four examples, each with one input feature and one target value.
x = torch.tensor([[0.], [1.], [2.], [3.]])
y = torch.tensor([[2.], [5.], [8.], [11.]])
loss_fn = torch.nn.MSELoss()
learning_rate = 0.05

print("Inputs:", x.flatten().tolist())
print("Targets:", y.flatten().tolist())
print("Input shape:", x.shape)
print("Target shape:", y.shape)


<a id="first-update"></a>
## Lesson 01: compare the first SGD and Adam updates

**Purpose:** observe how two optimizers use identical fresh gradients to choose different updates.

Create two separate models with the same initial weight and bias. Give SGD one model's parameters and Adam the other model's parameters. Both optimizers start with fresh state and use learning rate `0.05`. We apply one update using all four examples.

The same initial model values, data, and loss should give the same weight gradient, `-20`, and bias gradient, `-10`. Plain SGD scales each current gradient directly by the learning rate. Adam uses a different update procedure that scales gradients using its stored information about their direction and size.

We will inspect the two actual parameter changes rather than assume that identical learning-rate settings give identical steps. Leave Adam's other settings at their defaults for now; the notebook uses zero weight decay. Later lessons will introduce its stored state gradually.

This experiment describes one update at these settings. Longer training and validation results will help compare the optimizers in later lessons.

The cell recreates both models and optimizers each time, so rerunning it repeats the same first-update comparison.

**Before running:** should the two models have the same fresh gradients? For plain SGD, predict the weight and bias after its update.

**My prediction and reason:** _write here_


In [ ]:
# Independent models let us compare optimizers from the same starting point.
sgd_model = torch.nn.Linear(1, 1)
adam_model = torch.nn.Linear(1, 1)
with torch.no_grad():
    for current_model in [sgd_model, adam_model]:
        current_model.weight.fill_(1.)
        current_model.bias.fill_(0.)

# Register each optimizer with its own model's parameters.
sgd_optimizer = torch.optim.SGD(
    sgd_model.parameters(), lr=learning_rate, momentum=0, weight_decay=0
)
adam_optimizer = torch.optim.Adam(
    adam_model.parameters(), lr=learning_rate, weight_decay=0
)

# Run the same familiar training sequence once for each model.
for name, current_model, current_optimizer in [
    ("SGD", sgd_model, sgd_optimizer),
    ("Adam", adam_model, adam_optimizer),
]:
    current_model.train()
    weight_before = current_model.weight.item()
    bias_before = current_model.bias.item()
    current_optimizer.zero_grad(set_to_none=True)
    predictions = current_model(x)
    loss = loss_fn(predictions, y)
    loss.backward()  # Both models calculate gradients before their update.

    print(name)
    print("  Loss before update:", loss.item())
    print("  Weight gradient:", current_model.weight.grad.item())
    print("  Bias gradient:", current_model.bias.grad.item())
    current_optimizer.step()  # The two optimizers choose updates differently.

    # Evaluate the new parameters without calculating another gradient.
    with torch.no_grad():
        loss_after = loss_fn(current_model(x), y).item()
    print("  Weight after update:", current_model.weight.item())
    print("  Bias after update:", current_model.bias.item())
    print("  Weight increase:", current_model.weight.item() - weight_before)
    print("  Bias increase:", current_model.bias.item() - bias_before)
    print("  Loss after update:", loss_after)


**My observation and explanation:** _write here_

Compare gradients first, then compare the parameter changes. The gradients describe the same starting loss calculation. The optimizer determines how those gradients become a parameter update.

In this first Adam update, both gradients are nonzero and the two parameter increases are approximately the same size. That is a feature of this initial update; it does not mean Adam always gives every parameter an identical change on later updates.



<a id="first-step-size"></a>
## Lesson 02: explain the first Adam step with small numbers

**Purpose:** explain the observed first-step sizes without introducing Adam's full internal calculation yet.

In Lesson 01, the weight gradient was `-20` and the bias gradient was `-10`. SGD multiplied those gradients directly by learning rate `0.05` when choosing the changes. Adam used different scaling.

For this first update, with a fresh Adam optimizer, nonzero gradients, and zero weight decay, a helpful approximation is to divide each gradient by its own size, ignoring the sign only in the divisor:

- Weight: divide `-20` by `20`, giving `-1`.
- Bias: divide `-10` by `10`, also giving `-1`.

Apply learning rate `0.05` to those scaled values and subtract the result. Both parameters increase by approximately `0.05`. The code below shows that arithmetic next to the changes made by the real Adam optimizer.

This cell explains Lesson 01's first step. It does not perform another optimizer update or modify the stored gradients. Run it immediately after Lesson 01; if you have trained further, rerun Lesson 01 first to recreate this starting comparison.

Adam's actual implementation also uses stored averages, a correction for their initial empty history, and a small numerical-stability setting. We will introduce those gradually. The simplified calculation here explains these nonzero gradients at the first step; later updates use the accumulated history and can have different sizes.

**Before running:** what do `-20 / 20` and `-10 / 10` give? Should the approximate parameter changes be equal in this first-step example?

**My prediction and reason:** _write here_


In [ ]:
# Explain the first Adam update already performed in Lesson 01.
# These are arithmetic calculations only; we do not call backward() or step().
for name, parameter, value_before_first_step in [
    ("weight", adam_model.weight, 1.0),
    ("bias", adam_model.bias, 0.0),
]:
    original_gradient = parameter.grad.item()
    gradient_size = abs(original_gradient)  # Size without its positive/negative sign.
    scaled_value = original_gradient / gradient_size  # Both gradients here are nonzero.
    approximate_change = -learning_rate * scaled_value
    actual_change = parameter.item() - value_before_first_step

    print(name)
    print("  Original gradient:", original_gradient)
    print("  Gradient size:", gradient_size)
    print("  Scaled value for this first step:", scaled_value)
    print("  Approximate parameter increase:", approximate_change)
    print("  Actual Adam increase:", actual_change)
    print("  Stored gradient remains:", parameter.grad.item())


**My observation and explanation:** _write here_

The sign still matters: these negative gradients produced increases. The different original magnitudes did not produce proportionally different Adam changes on this first step.

The stored gradients themselves are unchanged. This cell illustrates how the optimizer can use the same gradient values differently when choosing an update. It does not replace the real optimizer with this simplified arithmetic for later training.



<a id="adam-memory"></a>
## Lesson 03: inspect Adam's stored history

**Purpose:** separate current parameter gradients from the information Adam remembers between updates.

Momentum SGD had a remembered buffer. Adam keeps two running memories for each parameter:

- A running average of gradients, which retains information about their signs and values.
- A running average of squared gradients, which retains information about their size. Squaring makes positive and negative gradients contribute positive size values.

The optimizer combines this information when choosing its update. We will inspect the stored values first and explain how they change in later lessons.

In this PyTorch optimizer state, the entries are:

| Entry | Meaning |
|---|---|
| `step` | Number of Adam updates recorded for this parameter |
| `exp_avg` | Running gradient average |
| `exp_avg_sq` | Running squared-gradient average |

These averages are different from the raw gradients in `parameter.grad`. Their weighting settings determine how much new information is mixed into the stored values; we will explore that gradually.

Run this cell after Lessons 01–02, while our Adam model has completed one update. Inspect its history, clear the current gradients, then inspect the history again. The cell does not calculate a new gradient or call `step()`.

The history and parameter values should stay unchanged when gradients are cleared. To recreate the first-update state after later training, rerun Lesson 01, then Lesson 02 before this cell.

**Before running:** should `zero_grad()` erase Adam's remembered averages, or only the current gradients?

**My prediction and reason:** _write here_


In [ ]:
# Inspect the Adam optimizer that completed one update in Lesson 01.
print("Before clearing current gradients")
for name, parameter in adam_model.named_parameters():
    stored = adam_optimizer.state[parameter]
    print(name)
    print("  Current gradient:", parameter.grad.item() if parameter.grad is not None else None)
    print("  State entry names:", list(stored.keys()))
    print("  Updates recorded:", int(stored["step"].item()))
    print("  Remembered gradient average:", stored["exp_avg"].flatten().tolist())
    print("  Remembered squared-gradient average:", stored["exp_avg_sq"].flatten().tolist())

# Clear parameter.grad without erasing the optimizer's two running averages.
adam_optimizer.zero_grad(set_to_none=True)

print("After clearing current gradients")
for name, parameter in adam_model.named_parameters():
    stored = adam_optimizer.state[parameter]
    print(name)
    print("  Current gradient:", parameter.grad)
    print("  Updates recorded:", int(stored["step"].item()))
    print("  Remembered gradient average:", stored["exp_avg"].flatten().tolist())
    print("  Remembered squared-gradient average:", stored["exp_avg_sq"].flatten().tolist())

print("Weight still:", adam_model.weight.item())
print("Bias still:", adam_model.bias.item())


**My observation and explanation:** _write here_

Clearing gradients prepares us to calculate fresh ones for the next batch. Adam's remembered averages are kept so the next update can use their history. Reading the history or clearing gradients does not increase the stored step count. Repeating this inspection leaves gradients at `None` and preserves the same history.

The raw first gradients were `-20` and `-10`, while the stored averages have different values because Adam mixes new information into running averages. We will explain those weights using small numbers in the next lesson.



<a id="gradient-average"></a>
## Lesson 04: build the remembered gradient average

**Purpose:** explain the `exp_avg` values by separating the old and fresh contributions.

With Adam's default first memory setting, keep `0.9` of the old gradient average and add `0.1` of the fresh gradient. These are 90% and 10% respectively.

On the first update, the old average was zero. For the weight gradient `-20`, the old contribution was zero and the fresh contribution was `0.1 * (-20)`, giving `-2`. For bias gradient `-10`, the same idea gave `-1`.

For a small what-if example, suppose this weight's next gradient were `-10`. Its old average is now `-2`. The old contribution would be `0.9 * (-2) = -1.8`, and the fresh contribution would be `0.1 * (-10) = -1`. Adding them gives a candidate new average of `-2.8`.

That `-10` is an illustrative next weight gradient, not a new gradient calculated from our current model and batch. We are doing small code calculations to understand the memory rule. The cell does not train the model or change the real optimizer history.

`betas` is the optimizer setting containing the weights for its two running memories. We read only its first value here, which controls the gradient average. We will explain the squared-gradient memory separately.

Run this after Lesson 03 while Adam has completed one update. Repeating this calculation leaves the optimizer state unchanged. To recreate the one-update state after later training, rerun Lesson 01 and then Lesson 03.

**Before running:** calculate the old contribution and fresh contribution separately, then add them. Is the candidate average just the fresh gradient?

**My prediction and reason:** _write here_


In [ ]:
# Read the setting for Adam's running gradient average.
old_share = adam_optimizer.param_groups[0]["betas"][0]
fresh_share = 1.0 - old_share
print(f"Share kept from old average: {old_share:.1f}")
print(f"Share added from fresh gradient: {fresh_share:.1f}")

# Reconstruct the first remembered weight-gradient average from known values.
initial_average = 0.0
first_weight_gradient = -20.0
first_old_part = old_share * initial_average
first_fresh_part = fresh_share * first_weight_gradient
first_average = first_old_part + first_fresh_part
print(f"First old contribution: {first_old_part:.4f}")
print(f"First fresh contribution: {first_fresh_part:.4f}")
print(f"First gradient average: {first_average:.4f}")

# Read the real average after Lesson 01's first update.
stored_average = adam_optimizer.state[adam_model.weight]["exp_avg"].item()
print(f"Actual stored weight average: {stored_average:.4f}")

# What-if only: this is not a gradient recalculated from the current batch.
illustrative_next_gradient = -10.0
old_part = old_share * stored_average
fresh_part = fresh_share * illustrative_next_gradient
candidate_average = old_part + fresh_part
print(f"What-if old contribution: {old_part:.4f}")
print(f"What-if fresh contribution: {fresh_part:.4f}")
print(f"What-if next average: {candidate_average:.4f}")

# The real optimizer has not been updated by these arithmetic calculations.
print("Actual average still:", adam_optimizer.state[adam_model.weight]["exp_avg"].item())
print("Actual updates still:", int(adam_optimizer.state[adam_model.weight]["step"].item()))


**My observation and explanation:** _write here_

The candidate average combines remembered and fresh information. It is neither the previous average alone nor the fresh gradient alone. Both contributions can be understood with multiplication and addition before studying further optimizer details.

The gradient average is one ingredient of Adam's parameter update. The squared-gradient average also contributes to how that update is scaled, so this running-average calculation alone is not the complete parameter-update rule.



<a id="squared-average"></a>
## Lesson 05: build the remembered squared-gradient average

**Purpose:** explain the positive `exp_avg_sq` values by squaring each gradient before mixing it into memory.

The weight's first gradient was `-20`. Squaring it gives `(-20) * (-20) = 400`. The bias's first gradient was `-10`, whose square is `100`. These squares keep information about size without a negative sign.

Adam uses a separate setting for this second memory. By default, keep `0.999` of the old squared-gradient average and add `0.001` of the new squared gradient. `0.001` means one thousandth: `400 * 0.001` gives `0.4`, just like `400 / 1000`.

On the first update, old memory is zero. Thus the first weight memory becomes about `0.4`, and bias memory becomes about `0.1`. Those are the values we inspected in Lesson 03.

| Memory | Share of old memory kept | Share of fresh information added |
|---|---:|---:|
| Gradient average (`exp_avg`) | `0.9` | `0.1` of the gradient |
| Squared-gradient average (`exp_avg_sq`) | `0.999` | `0.001` of the squared gradient |

For a what-if next weight gradient of `-10`, the fresh square would be `100`. The old contribution would be about `0.999 * 0.4 = 0.3996`, and the fresh contribution `0.001 * 100 = 0.1`. Their sum is about `0.4996`.

That what-if gradient is illustrative. This cell reads the actual first-update memory and performs small calculations, without training or changing the real stored values. Run it after Lesson 04, while the real optimizer still has one completed update. To recreate that state after later training, rerun Lesson 01 and then Lesson 03.

**Before running:** why is the squared-gradient average positive even when the raw gradients are negative? Calculate the first contributions `0.001 * 400` and `0.001 * 100`.

**My prediction and reason:** _write here_


In [ ]:
# Read the setting for Adam's squared-gradient average: the second beta.
old_squared_share = adam_optimizer.param_groups[0]["betas"][1]
fresh_squared_share = 1.0 - old_squared_share
print(f"Share kept from old squared average: {old_squared_share:.3f}")
print(f"Share added from fresh squared gradient: {fresh_squared_share:.3f}")

# Reconstruct the first squared-gradient memories from the known first gradients.
for name, parameter, first_gradient in [
    ("weight", adam_model.weight, -20.0),
    ("bias", adam_model.bias, -10.0),
]:
    first_squared_gradient = first_gradient * first_gradient
    first_squared_average = old_squared_share * 0.0 + fresh_squared_share * first_squared_gradient
    actual_squared_average = adam_optimizer.state[parameter]["exp_avg_sq"].item()
    print(name)
    print("  First squared gradient:", first_squared_gradient)
    print(f"  Calculated first squared average: {first_squared_average:.4f}")
    print(f"  Actual first squared average: {actual_squared_average:.4f}")

# What-if only: imagine the next weight gradient were -10.
old_squared_average = adam_optimizer.state[adam_model.weight]["exp_avg_sq"].item()
illustrative_next_gradient = -10.0
fresh_squared_gradient = illustrative_next_gradient * illustrative_next_gradient
old_squared_part = old_squared_share * old_squared_average
fresh_squared_part = fresh_squared_share * fresh_squared_gradient
candidate_squared_average = old_squared_part + fresh_squared_part
print(f"What-if old contribution: {old_squared_part:.4f}")
print(f"What-if fresh contribution: {fresh_squared_part:.4f}")
print(f"What-if next squared average: {candidate_squared_average:.4f}")

# Reading and calculating do not overwrite Adam's real history.
print("Actual weight squared average still:", adam_optimizer.state[adam_model.weight]["exp_avg_sq"].item())
print("Actual updates still:", int(adam_optimizer.state[adam_model.weight]["step"].item()))


**My observation and explanation:** _write here_

Adam's two memories keep different information and use different weighting settings. The gradient average retains signed values. The squared-gradient average tracks size information that is nonnegative.

The remembered squared value is not the parameter's current gradient or its parameter value. The optimizer uses both running averages when scaling the actual update. The real state remains at the first update throughout these illustrative calculations.



<a id="second-update"></a>
## Lesson 06: perform the actual second Adam update

**Purpose:** calculate fresh gradients at the current parameters, update both running memories, and apply the next real Adam step.

Our real Adam model has completed one update. Its weight is about `1.05` and bias about `0.05`. Lessons 02–05 inspected or explained that state without making another update.

Now use the same batch, but calculate new predictions and gradients using the current parameter values. Do not substitute the illustrative gradient `-10` from our what-if examples. The actual gradient must come from this model's loss and `backward()`.

Print the old memories, clear current gradients, calculate fresh ones, and call `step()`. Adam combines the old memories with this fresh information and chooses the new parameter changes. Print its state again afterward to see the count become two.

Run this cell once after Lessons 01–05. Rerunning it alone performs a further update. To repeat the original second-update experiment, rerun Lesson 01 to recreate its first-update model, then run this cell. The intervening explanatory lessons do not change that model's values or stored history.

**Before running:** should the fresh gradients necessarily equal the original `-20` and `-10`? What should the stored update count be after this step?

**My prediction and reason:** _write here_


In [ ]:
# Continue the same Adam model and optimizer after their first actual update.
weight_before = adam_model.weight.item()
bias_before = adam_model.bias.item()
print("Updates before step:", int(adam_optimizer.state[adam_model.weight]["step"].item()))
for name, parameter in adam_model.named_parameters():
    stored = adam_optimizer.state[parameter]
    print(name, "old gradient average:", stored["exp_avg"].item())
    print(name, "old squared-gradient average:", stored["exp_avg_sq"].item())

# Calculate new gradients from the current model values and the actual batch.
adam_optimizer.zero_grad(set_to_none=True)
fresh_predictions = adam_model(x)
fresh_loss = loss_fn(fresh_predictions, y)
fresh_loss.backward()
print("Fresh weight gradient:", adam_model.weight.grad.item())
print("Fresh bias gradient:", adam_model.bias.grad.item())
print("Loss before this update:", fresh_loss.item())

# One real update changes Adam's memories as well as model parameter values.
adam_optimizer.step()

for name, parameter in adam_model.named_parameters():
    stored = adam_optimizer.state[parameter]
    print(name, "new gradient average:", stored["exp_avg"].item())
    print(name, "new squared-gradient average:", stored["exp_avg_sq"].item())
print("Updates after step:", int(adam_optimizer.state[adam_model.weight]["step"].item()))
print("Weight after step:", adam_model.weight.item())
print("Bias after step:", adam_model.bias.item())
print("Weight increase this step:", adam_model.weight.item() - weight_before)
print("Bias increase this step:", adam_model.bias.item() - bias_before)

# Measure the new loss without calculating another gradient or update.
with torch.no_grad():
    loss_after_second_update = loss_fn(adam_model(x), y).item()
print("Loss after this update:", loss_after_second_update)


**My observation and explanation:** _write here_

Compare the fresh gradients with the first update's gradients. The loss was recalculated at changed parameter values, so the gradients changed. The new remembered values now combine information from two actual updates.

The actual fresh weight gradient differs from the illustrative `-10` used in Lessons 04–05. Consequently, the real new memory values differ from those what-if candidates. Reading or calculating examples did not replace the optimizer's real state.

Adam's parameter changes use its running memories and corrections for their initial zero values. We will explain that connection with small code calculations next. Its step is more than multiplying the raw gradient by the learning rate, or multiplying the uncorrected gradient average alone by the learning rate.



<a id="memory-to-step"></a>
## Lesson 07: connect the first-step memories to the parameter change

**Purpose:** connect the two stored memories to the approximately `0.05` first weight increase, using one small trace.

A positive square root reverses squaring: `20 * 20` gives `400`, so the positive square root of `400` is `20`. In this cell, `math.sqrt()` performs that operation.

The first weight gradient was `-20`. Adam's stored gradient average became `-2`, and its stored squared-gradient average became about `0.4`. They started from zero, so these first averages contain only the weighted part of the new information. Adam corrects for that early zero initialization before using them in its parameter update.

For the first step, undo the initial weighting:

- Corrected gradient average: divide `-2` by `0.1`, giving `-20`.
- Corrected squared-gradient average: divide `0.4` by `0.001`, giving about `400`.
- Take the positive square root of the corrected squared value, giving about `20`.
- Divide the corrected gradient average by that size: about `-20 / 20 = -1`.
- Scale by learning rate `0.05` and subtract the result, increasing the weight by about `0.05`.

Adam also adds a very small stability value to the divisor. It helps make division safe when the size is near zero. We read the optimizer's current setting instead of tuning it here.

This is a first-step explanation. On later steps, Adam's correction factors depend on the recorded update count as well as its memory settings. Continue using the real optimizer for training; this small trace illustrates what it did on the first step.

The cell creates a separate demonstration model and optimizer, recreates one update, and inspects only the weight. It leaves our ongoing Adam model from Lesson 06 at its two-update state. Repeating this cell recreates the same independent demonstration.

**Before running:** what positive square root should the corrected squared value near `400` have? Should the reconstructed change be close to the real first Adam weight change?

**My prediction and reason:** _write here_


In [ ]:
import math

# Separate first-step demonstration; keep our ongoing adam_model unchanged.
first_step_model = torch.nn.Linear(1, 1)
with torch.no_grad():
    first_step_model.weight.fill_(1.)
    first_step_model.bias.fill_(0.)
first_step_optimizer = torch.optim.Adam(
    first_step_model.parameters(), lr=learning_rate, weight_decay=0
)
weight_before_demo = first_step_model.weight.item()
first_step_optimizer.zero_grad(set_to_none=True)
demo_loss = loss_fn(first_step_model(x), y)
demo_loss.backward()
first_step_optimizer.step()

# Read the memories created by that actual first update.
stored = first_step_optimizer.state[first_step_model.weight]
settings = first_step_optimizer.param_groups[0]
gradient_memory_setting, squared_memory_setting = settings["betas"]
stored_gradient_average = stored["exp_avg"].item()
stored_squared_average = stored["exp_avg_sq"].item()

# On step 1, compensate for the memories starting at zero.
corrected_gradient_average = stored_gradient_average / (1.0 - gradient_memory_setting)
corrected_squared_average = stored_squared_average / (1.0 - squared_memory_setting)
gradient_size = math.sqrt(corrected_squared_average)

# Read Adam's tiny stability value and reconstruct its first weight change.
stability_value = settings["eps"]
scaled_direction = corrected_gradient_average / (gradient_size + stability_value)
reconstructed_change = -settings["lr"] * scaled_direction
actual_change = first_step_model.weight.item() - weight_before_demo

print("Stored gradient average:", stored_gradient_average)
print("Stored squared-gradient average:", stored_squared_average)
print(f"Corrected gradient average: {corrected_gradient_average:.4f}")
print(f"Corrected squared-gradient average: {corrected_squared_average:.4f}")
print(f"Positive square root: {gradient_size:.4f}")
print(f"Scaled direction: {scaled_direction:.4f}")
print("Reconstructed weight increase:", reconstructed_change)
print("Actual first Adam weight increase:", actual_change)
print("Ongoing optimizer updates still:", int(adam_optimizer.state[adam_model.weight]["step"].item()))


**My observation and explanation:** _write here_

The reconstructed change closely matches the real first update. Small differences come from rounding and the tensor's floating-point storage. We used both memories after correcting their initial weighting, then took a square root to convert the squared-size value back into a size.

The memories determine the update through this scaling; neither stored average is itself the new parameter value. The separate demonstration did not perform a third update on our ongoing model.



<a id="longer-comparison"></a>
## Lesson 08: compare longer training and validation runs

**Purpose:** compare outcomes at several equal update budgets, and observe whether training loss always decreases.

Adam's history and size scaling can help smooth updates in some problems. Smaller steps can also help avoid overshooting. The behavior depends on the data, optimizer, and settings; inspect the actual loss rather than assume every update improves it.

Start independent SGD and Adam runs at weight `1` and bias `0`. Use the same training data, full-batch loss, learning rate `0.05`, and 200-update limit. Each update uses all four training examples. The ongoing Adam model from Lesson 06 remains separate.

Use unseen validation inputs `[4, 5]` and targets `[14, 17]` to measure predictions beyond the training inputs. Evaluate training and validation loss after each update without calculating extra gradients. Compare equal budgets of 10, 100, and 200 updates.

Also record the first step where full training loss increases from the preceding step, if any. The same examples are evaluated each time, so this is a change in the model's loss, not an effect of evaluating a different batch.

This fixed learning-rate comparison illustrates these runs. Optimizers can need different learning-rate settings on other problems. Validation over appropriate update budgets helps assess performance; one early result does not establish a universal ranking.

The cell recreates both comparison models and optimizers, so rerunning it repeats the experiment. It requires Setup's data and loss function. The `first_rise` record describes an observed increase, not an early-stopping condition.

**Before running:** will the first-update winner necessarily have the lowest validation loss at all later budgets? Does using Adam guarantee that training loss decreases on every update?

**My prediction and reason:** _write here_


In [ ]:
# Validation examples are used only for evaluation in this experiment.
validation_x = torch.tensor([[4.], [5.]])
validation_y = torch.tensor([[14.], [17.]])
comparison_updates = 200
report_updates = {1, 2, 5, 10, 20, 50, 80, 100, 200}
comparison_history = {}

for optimizer_name in ["SGD", "Adam"]:
    # Each run starts from identical known values and fresh optimizer history.
    comparison_model = torch.nn.Linear(1, 1)
    with torch.no_grad():
        comparison_model.weight.fill_(1.)
        comparison_model.bias.fill_(0.)
    if optimizer_name == "SGD":
        comparison_optimizer = torch.optim.SGD(
            comparison_model.parameters(), lr=0.05, momentum=0, weight_decay=0
        )
    else:
        comparison_optimizer = torch.optim.Adam(
            comparison_model.parameters(), lr=0.05, weight_decay=0
        )

    comparison_model.train()
    with torch.no_grad():
        previous_training_loss = loss_fn(comparison_model(x), y).item()
    first_rise = None
    history = {}
    print(optimizer_name)

    for update in range(1, comparison_updates + 1):
        comparison_optimizer.zero_grad(set_to_none=True)
        training_loss = loss_fn(comparison_model(x), y)
        training_loss.backward()
        comparison_optimizer.step()

        # Measure both losses at the same current parameters after this update.
        with torch.no_grad():
            training_after = loss_fn(comparison_model(x), y).item()
            validation_after = loss_fn(comparison_model(validation_x), validation_y).item()
        history[update] = {"training": training_after, "validation": validation_after}

        # Record a loss increase for discussion; keep training to the fixed budget.
        if first_rise is None and training_after > previous_training_loss:
            first_rise = {
                "update": update,
                "loss_before": previous_training_loss,
                "loss_after": training_after,
            }
        previous_training_loss = training_after

        if update in report_updates:
            print(f"  Update {update:3d} | training={training_after:.8f} | validation={validation_after:.8f}")

    comparison_history[optimizer_name] = history
    print("  First training-loss increase:", first_rise)
    print("  Final weight:", comparison_model.weight.item())
    print("  Final bias:", comparison_model.bias.item())

# Compare validation losses at matching numbers of parameter updates.
print("Optimizer | Validation at 10 | Validation at 100 | Validation at 200")
for optimizer_name, history in comparison_history.items():
    print(f"{optimizer_name:9s} | {history[10]['validation']:.8f} | "
          f"{history[100]['validation']:.8f} | {history[200]['validation']:.8f}")


**My observation and explanation:** _write here_

Compare each optimizer at the same update budget. Explain whether the ranking changes between 100 and 200 updates. Inspect the recorded first rise to assess the claim that Adam necessarily gives a steadily decreasing training loss.

A temporary rise in training loss does not by itself establish overfitting. The training and validation trends, chosen settings, and data matter. For this simple clean-data exercise, both runs eventually fit the supplied rule closely.

The smaller final values in this experiment are observations for these settings. Appropriate independent validation and consistent comparison budgets guide choices on a new problem.

<a id="chapter-08-review"></a>
## Chapter 08 recap

All eight lessons have been run and reviewed in the mentoring conversation.

- SGD and Adam can receive the same gradients and choose different parameter changes. The same learning-rate setting does not imply the same step size.
- Adam stores a running gradient average, a running squared-gradient average, and an update counter for each parameter that receives updates.
- `zero_grad()` clears current gradients while preserving optimizer history. Each training update still needs fresh predictions, loss, and gradients.
- With the default settings, the gradient average keeps 90% of its old value and adds 10% of the fresh gradient. The squared-gradient average keeps 99.9% of its old value and adds 0.1% of the fresh squared gradient.
- Adam corrects the initial weighting of its averages and uses a square root to turn the squared-size estimate into a size. Our separate first-step trace reconstructed the observed weight increase without changing the ongoing model.
- The approximately `0.05` increases in both parameters were observations for our first Adam step with nonzero gradients and no weight decay. Later updates depend on the evolving history.
- Adam does not guarantee that loss decreases on every update. In the comparison, its first training-loss increase occurred at update 61; SGD's training loss did not increase during these 200 updates.
- Compare validation loss at matching update budgets. The winner can change as training continues, and appropriate learning rates can differ between optimizers.

### Recorded experiment: same data and learning rate `0.05`

After the first update, SGD had weight `2`, bias `0.5`, and training loss `10.25`. Adam had weight approximately `1.05`, bias `0.05`, and training loss `28.518753`. Both started with weight gradient `-20` and bias gradient `-10`.

| Optimizer | Validation loss after 100 updates | Validation loss after 200 updates |
|---|---:|---:|
| SGD | 0.00232331 | 0.00000554 |
| Adam | 0.00350201 | 0.00000014 |

For these settings, select SGD at a 100-update budget and Adam at a 200-update budget using validation loss. This experiment does not establish a universal optimizer ranking. A temporary training-loss increase alone does not prove overfitting.

Next, introduce weight decay through a small programming experiment before meeting AdamW.

## Documentation

- [PyTorch: Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)
- [PyTorch: SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)
